In [1]:
import torch
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

# =========================
# SmolVLA fine-tune on a small LIBERO dataset and push to HF Hub
# Minimal-time pipeline for Kaggle T4 / multi-GPU
# =========================

# ---------- 0) User settings ----------
HF_MODEL_REPO = "kgaero/smolvla-libero-kaggle-test"
DATASET_REPO = "lerobot/libero_spatial_image"
OUTPUT_DIR = "/content/outputs/train/smolvla_libero_quick_multi"
JOB_NAME = "smolvla_libero_quick_multi"

# Safer for T4 x2 without AMP
BATCH_SIZE = 2          # start with 2; increase later only if stable
STEPS = 300
SAVE_FREQ = 100
EVAL_FREQ = 100

# ---------- 1) Optional: verify GPU ----------
!nvidia-smi || true

# ---------- 2) Reset working directory ----------
%cd /content

# ---------- 3) Fresh clone ----------
!rm -rf /content/lerobot
!git clone https://github.com/huggingface/lerobot.git /content/lerobot

# ---------- 4) Install ----------
%cd /content/lerobot
!python -m pip install --upgrade pip wheel
!pip install -e ".[smolvla]"
!pip install -U accelerate

# ---------- 5) Login to Hugging Face ----------
from huggingface_hub import login
import os
import subprocess
import torch

token = ""
login(token=token, add_to_git_credential=True)

!git config --global credential.helper store
!hf auth whoami

# ---------- 6) Reduce tokenizer warnings ----------
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---------- 7) Detect GPU count ----------
NUM_GPUS = torch.cuda.device_count()
print(f"Detected {NUM_GPUS} GPU(s)")

# ---------- 8) Run training ----------
if NUM_GPUS > 1:
    train_cmd = f"""
    cd /content/lerobot && accelerate launch \
      --multi_gpu \
      --num_processes={NUM_GPUS} \
      --mixed_precision=fp16 \
      $(which lerobot-train) \
      --policy.path=lerobot/smolvla_base \
      --dataset.repo_id={DATASET_REPO} \
      --batch_size={BATCH_SIZE} \
      --steps={STEPS} \
      --output_dir={OUTPUT_DIR} \
      --job_name={JOB_NAME} \
      --policy.device=cuda \
      --wandb.enable=false \
      --policy.repo_id={HF_MODEL_REPO} \
      --policy.push_to_hub=true \
      --save_freq={SAVE_FREQ} \
      --eval_freq={EVAL_FREQ} \
      --rename_map='{{"observation.images.image":"observation.images.camera1","observation.images.wrist_image":"observation.images.camera2"}}' \
      --policy.empty_cameras=1
    """
else:
    train_cmd = f"""
    cd /content/lerobot && lerobot-train \
      --policy.path=lerobot/smolvla_base \
      --dataset.repo_id={DATASET_REPO} \
      --batch_size={BATCH_SIZE} \
      --steps={STEPS} \
      --output_dir={OUTPUT_DIR} \
      --job_name={JOB_NAME} \
      --policy.device=cuda \
      --wandb.enable=false \
      --policy.repo_id={HF_MODEL_REPO} \
      --policy.push_to_hub=true \
      --save_freq={SAVE_FREQ} \
      --eval_freq={EVAL_FREQ} \
      --rename_map='{{"observation.images.image":"observation.images.camera1","observation.images.wrist_image":"observation.images.camera2"}}' \
      --policy.empty_cameras=1
    """

print(train_cmd)
print("Starting training...")
!{train_cmd}

print("\nDone.")
print("Your pushed model repo should be:")
print(HF_MODEL_REPO)
print("\nOn your desktop, download with:")
print(f"huggingface-cli download {HF_MODEL_REPO} --local-dir $HOME/robotics/libero_smolvla_eval/policies/smolvla_libero_quick")

GPU count: 2
0 Tesla T4
1 Tesla T4
Thu Mar 19 05:03:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
A new version of huggingface_hub (1.7.1) is available! You are using version 1.4.1.
To update, run: pip install -U huggingface_hub

user:  kgaero
Detected 2 GPU(s)

    cd /content/lerobot && accelerate launch       --multi_gpu       --num_processes=2       --mixed_precision=fp16       $(which lerobot-train)       --policy.path=lerobot/smolvla_base       --dataset.repo_id=lerobot/libero_spatial_image       --batch_size=2       --steps=300       --output_dir=/content/outputs/train/smolvla_libero_quick_multi       --job_name=smolvla_libero_quick_multi       --policy.device=cuda       --wandb.enable=fa